In [6]:
import numpy as np
import json
from tqdm import tqdm
import h5py

from rdkit import Chem
from rdkit.Chem import AllChem
from unimol_tools import UniMolRepr

import torch
from transformers import T5Tokenizer, T5EncoderModel, T5ForConditionalGeneration

from e3fp.fingerprint.fprint import Fingerprint
from e3fp.conformer.generate import generate_conformers
import pandas as pd
import deepchem as dc

In [17]:
# with open(r"E:\pUGTdb\20251128_collection\download_PF00201_taxonomy33090\step2_get_3D_Protein\smiles_to_id.json") as f:
#     smiles_to_id_o = json.load(f)
with open(r"E:\pUGTdb\20251128_collection\download_PF00201_taxonomy33090\step2_get_3D_Protein\DATASETS\id2smile.json") as f:
    smiles_to_id_o = json.load(f)

In [18]:
smiles_to_id_o

{'CC(C)=CCC[C@](C)(O)[C@H]1CC[C@]2(C)[C@@H]1[C@H](O)C[C@@H]1[C@@]3(C)CC[C@H](O)C(C)(C)[C@@H]3CC[C@@]21C': 'SMILES_0',
 'CC(C)=CCC[C@](C)(O)[C@H]1CC[C@]2(C)[C@@H]1[C@H](O)C[C@@H]1[C@@]3(C)CC[C@H](O)C(C)(C)[C@@H]3[C@@H](O)C[C@@]21C': 'SMILES_1',
 'CC(C)=CCC[C@](C)(O[C@@H]1O[C@H](CO)[C@@H](O)[C@H](O)[C@H]1O)[C@H]1CC[C@]2(C)[C@@H]1[C@H](O)C[C@@H]1[C@@]3(C)CC[C@H](O)C(C)(C)[C@@H]3[C@@H](O)C[C@@]21C': 'SMILES_2',
 'Oc1cc([O-])c2cc([O-])c([o+]c2c1)-c1ccc(O)c(O)c1': 'SMILES_3',
 'C1=C(C=C2[O+]=C(C(=CC2=C1[O-])[O-])C=3C=C(C(O)=C(C3)O)O)O': 'SMILES_5',
 'C1=C(C=C2[O+]=C(C(=CC2=C1[O-])[O-])C=3C=C(C(O)=C(C3)OC)OC)O': 'SMILES_6',
 'C1=C(C=C2[O+]=C(C(=CC2=C1[O-])[O-])C=3C=CC(O)=CC3)O': 'SMILES_8',
 'C1=C(C=C2[O+]=C(C(=CC2=C1[O-])[O-])C=3C=C(C(O)=CC3)OC)O': 'SMILES_9',
 'C1=2C([O+]=C(C3=CC(=C(C(=C3)OC)O)O)C(=C1)[O-])=CC(=CC2[O-])O': 'SMILES_11',
 'C1=C(C(=[O+]C2=CC(=CC(=C12)[O-])O)C3=CC(=C(C(=C3)*)O)*)[O-]': 'SMILES_12',
 '[C@@]12(C)[C@]3([C@]([C@@]4(CC[C@@H](C(C4=CC3)(C)C)O)[H])([C@@H](C[C@]2(C)[C@]

# Normlize the SMILEs format

In [19]:
def standardize_smiles(smiles):
    """
    标准化 SMILES 字符串：
    1. 解析为 Mol 对象 (容错)
    2. 重新输出为标准 Canonical SMILES
    3. 强制保留手性 (Isomeric)
    """
    if not isinstance(smiles, str):
        return None
        
    # 1. 解析
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        print(f"警告: 无法解析的 SMILES: {smiles}")
        return None
    
    # (可选) 移除盐和溶剂，只保留最大的有机片段
    # 很多生物数据库里会有 ".Cl" 或 ".Na" 这种盐，通常需要去掉
    # 这里是一个简单的去盐逻辑：只取最长的那个片段
    # mol = max(Chem.GetMolFrags(mol, asMols=True), key=lambda m: m.GetNumAtoms())

    # 2. 输出为标准格式
    # isomericSmiles=True : 非常重要！保留 @/@@ 手性信息
    # canonical=True : RDKit 默认就是 True，保证唯一的原子编号顺序
    new_smiles = Chem.MolToSmiles(mol, canonical=True, isomericSmiles=True)
    
    return new_smiles

# # --- 测试 ---
# smi1 = "COC(=O)C1=CO[C@@H](O)[C@@H]2[C@@H](C)CC[C@H]12"
# smi2 = "[C@]123[C@]([C@]4([C@@]([C@](CCC4)(C)C(O[C@@H]5O[C@@H]([C@@H](O)[C@@H]([C@H]5O[C@@H]6O[C@@H]([C@@H](O)[C@@H]([C@H]6O)O)CO)O)CO)=O)(CC1)[H])C)(CC[C@@](C2)(O[C@H]7[C@H](O[C@@H]8O[C@@H]([C@@H](O)[C@@H]([C@H]8O)O)CO)[C@H]([C@H](O)[C@H](O7)CO)O[C@@H]9O[C@@H]([C@@H](O)[C@@H]([C@H]9O)O)CO)C(C3)=C)[H]"

# std_1 = standardize_smiles(smi1)
# std_2 = standardize_smiles(smi2)

# print(f"原始 1: {smi1}")
# print(f"标准 1: {std_1}")
# print("-" * 20)
# print(f"原始 2: {smi2}")
# print(f"标准 2: {std_2}")

In [20]:
smiles_to_id = {}
for v,k in smiles_to_id_o.items():
    std_v = standardize_smiles(v)
    smiles_to_id[v] = k

In [21]:
with open(r"E:\pUGTdb\20251128_collection\download_PF00201_taxonomy33090\step2_get_3D_Protein\smiles_to_id_normlize1.json",'w') as f:
    json.dump(smiles_to_id,f, indent=4)

# UniMol embedding

In [22]:
clf = UniMolRepr(
    data_type='molecule',
    remove_hs=False,
    # pretrained_model_path=r'F:\UniMol_features_PT\mol_pre_all_h_220816.pt',
    # pretrained_dict_path=r'F:\UniMol_features_PT\mol.dict.txt'
)

2026-01-08 22:26:57 | unimol_tools\models\unimol.py | 120 | INFO | Uni-Mol Tools | Loading pretrained weights from C:\Users\admin\.conda\envs\Enzyme_design\lib\site-packages\unimol_tools\weights\mol_pre_all_h_220816.pt


In [23]:
molecule_embedding = {}
for smiles, id_ in tqdm(smiles_to_id.items()):
    smiles_list = [smiles]
    unimol_repr = clf.get_repr(smiles_list, return_atomic_reprs=True)
    # CLS token repr
    # print("Molecular feats:",np.array(unimol_repr['cls_repr']).shape)
    # # atomic level repr, align with rdkit mol.GetAtoms()
    # print("Atom feats:",np.array(unimol_repr['atomic_reprs']).shape)
    molecule_embedding[id_] = np.array(unimol_repr['cls_repr'])

  0%|          | 0/154 [00:00<?, ?it/s]2026-01-08 22:26:59 | unimol_tools\data\conformer.py | 89 | INFO | Uni-Mol Tools | Start generating conformers...

0it [00:00, ?it/s]
1it [00:04,  4.85s/it]
2026-01-08 22:27:04 | unimol_tools\data\conformer.py | 93 | INFO | Uni-Mol Tools | Succeed to generate conformers for 100.00% of molecules.
2026-01-08 22:27:04 | unimol_tools\data\conformer.py | 95 | INFO | Uni-Mol Tools | Succeed to generate 3d conformers for 100.00% of molecules.

  1%|          | 1/154 [00:06<16:24,  6.44s/it]2026-01-08 22:27:05 | unimol_tools\data\conformer.py | 89 | INFO | Uni-Mol Tools | Start generating conformers...

0it [00:00, ?it/s]
1it [00:02,  2.95s/it]
2026-01-08 22:27:08 | unimol_tools\data\conformer.py | 93 | INFO | Uni-Mol Tools | Succeed to generate conformers for 100.00% of molecules.
2026-01-08 22:27:08 | unimol_tools\data\conformer.py | 95 | INFO | Uni-Mol Tools | Succeed to generate 3d conformers for 100.00% of molecules.

  1%|▏         | 2/154 [00:10<12

In [24]:
smiles_to_id

{'CC(C)=CCC[C@](C)(O)[C@H]1CC[C@]2(C)[C@@H]1[C@H](O)C[C@@H]1[C@@]3(C)CC[C@H](O)C(C)(C)[C@@H]3CC[C@@]21C': 'SMILES_0',
 'CC(C)=CCC[C@](C)(O)[C@H]1CC[C@]2(C)[C@@H]1[C@H](O)C[C@@H]1[C@@]3(C)CC[C@H](O)C(C)(C)[C@@H]3[C@@H](O)C[C@@]21C': 'SMILES_1',
 'CC(C)=CCC[C@](C)(O[C@@H]1O[C@H](CO)[C@@H](O)[C@H](O)[C@H]1O)[C@H]1CC[C@]2(C)[C@@H]1[C@H](O)C[C@@H]1[C@@]3(C)CC[C@H](O)C(C)(C)[C@@H]3[C@@H](O)C[C@@]21C': 'SMILES_2',
 'Oc1cc([O-])c2cc([O-])c([o+]c2c1)-c1ccc(O)c(O)c1': 'SMILES_3',
 'C1=C(C=C2[O+]=C(C(=CC2=C1[O-])[O-])C=3C=C(C(O)=C(C3)O)O)O': 'SMILES_5',
 'C1=C(C=C2[O+]=C(C(=CC2=C1[O-])[O-])C=3C=C(C(O)=C(C3)OC)OC)O': 'SMILES_6',
 'C1=C(C=C2[O+]=C(C(=CC2=C1[O-])[O-])C=3C=CC(O)=CC3)O': 'SMILES_8',
 'C1=C(C=C2[O+]=C(C(=CC2=C1[O-])[O-])C=3C=C(C(O)=CC3)OC)O': 'SMILES_9',
 'C1=2C([O+]=C(C3=CC(=C(C(=C3)OC)O)O)C(=C1)[O-])=CC(=CC2[O-])O': 'SMILES_11',
 'C1=C(C(=[O+]C2=CC(=CC(=C12)[O-])O)C3=CC(=C(C(=C3)*)O)*)[O-]': 'SMILES_12',
 '[C@@]12(C)[C@]3([C@]([C@@]4(CC[C@@H](C(C4=CC3)(C)C)O)[H])([C@@H](C[C@]2(C)[C@]

In [25]:
save_path = r"E:\pUGTdb\20251128_collection\download_PF00201_taxonomy33090\step2_get_3D_Protein\smiles_UniMol_embedding1.h5"
# 保存
with h5py.File(save_path, 'w') as f:
    for id_, embedding in molecule_embedding.items():
        f.create_dataset(id_, data=np.array(embedding))

# RDKit embedding

In [26]:
def get_rdkit_features(smiles, radius=2, nBits=2048):
    """
    提取 RDKit Morgan 指纹 (ECFP4)
    :param smiles_list: SMILES 字符串列表
    :return: Numpy 数组 (Batch_Size, nBits)
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        # 处理无效 SMILES，返回全 0 向量
        fp_array = np.zeros((nBits,), dtype=np.float32)
    else:
        # 生成位向量 (Bit Vector)
        fp_bit = AllChem.GetMorganFingerprintAsBitVect(
            mol, 
            radius=radius, 
            nBits=nBits, 
            useChirality=True  # 强烈建议开启手性
        )
        # 转换为 Numpy
        fp_array = np.zeros((0,), dtype=np.int8)
        AllChem.DataStructs.ConvertToNumpyArray(fp_bit, fp_array)
        fp_2d = fp_array.reshape(1, -1)
        fp_array = fp_2d.astype(np.float32)
        
            
    
    return fp_array

In [27]:
molecule_embedding_RDKit = {}
for smiles, id_ in tqdm(smiles_to_id.items()):
    rdkit_feats = get_rdkit_features(smiles)
    # 1. 提取 RDKit 特征
    molecule_embedding_RDKit[id_] = rdkit_feats

100%|██████████| 154/154 [00:00<00:00, 1810.50it/s]


In [28]:
save_path = r"E:\pUGTdb\20251128_collection\download_PF00201_taxonomy33090\step2_get_3D_Protein\smiles_RDKit_embedding1.h5"
# 保存
with h5py.File(save_path, 'w') as f:
    for id_, embedding in molecule_embedding_RDKit.items():
        f.create_dataset(id_, data=np.array(embedding))

# MolT5 embedding

In [29]:

class MolT5FeatureExtractor:
    def __init__(self, model_name="laituan245/molt5-base", device="cuda"):
        """
        初始化 MolT5 提取器
        model_name 可选: 
        - 'laituan245/molt5-small' (快速调试)
        - 'laituan245/molt5-base'  (推荐，平衡)
        - 'laituan245/molt5-large' (效果最好，显存要求高)
        """
        self.device = device
        print(f"正在加载 MolT5 模型: {model_name} ...")
        self.tokenizer = T5Tokenizer.from_pretrained(model_name, model_max_length=512)
        self.model = T5EncoderModel.from_pretrained(model_name).to(self.device)
        self.model.eval()

    def get_features(self, smiles_list):
        """
        提取 MolT5 Embedding (Mean Pooling)
        :return: Numpy 数组 (Batch_Size, Hidden_Dim) 
                 Base 模型通常是 768 维
        """
        # 1. Tokenize (SMILES -> Token IDs)
        inputs = self.tokenizer(
            smiles_list, 
            return_tensors="pt", 
            padding=True, 
            truncation=True,
            max_length=512
        ).to(self.device)

        with torch.no_grad():
            # 2. Forward Pass (只过 Encoder)
            outputs = self.model(**inputs)
            
            # last_hidden_state shape: [Batch, Seq_Len, Hidden_Dim]
            last_hidden_state = outputs.last_hidden_state
            
            # 3. Mean Pooling (将变长序列压缩为固定向量)
            # 注意：我们要忽略 padding 部分的影响
            attention_mask = inputs['attention_mask'] # [Batch, Seq_Len]
            
            # 扩展 mask 维度以匹配 hidden_state
            mask_expanded = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
            
            # 求和并除以真实长度
            sum_embeddings = torch.sum(last_hidden_state * mask_expanded, 1)
            sum_mask = torch.clamp(mask_expanded.sum(1), min=1e-9)
            
            mean_embeddings = sum_embeddings / sum_mask
            
        return mean_embeddings.cpu().numpy()

In [30]:
# 2. 提取 MolT5 特征
# 如果没有显卡，把 device 改为 'cpu'
device = "cuda" if torch.cuda.is_available() else "cpu"
molt5_extractor = MolT5FeatureExtractor(model_name="E:\molt5-small", device=device)

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


正在加载 MolT5 模型: E:\molt5-small ...


In [31]:
molecule_embedding_molt5 = {}
for smiles, id_ in tqdm(smiles_to_id.items()):
    molt5_feats = molt5_feats = molt5_extractor.get_features(smiles)
    # 1. 提取 RDKit 特征
    molecule_embedding_molt5[id_] = molt5_feats

100%|██████████| 154/154 [00:01<00:00, 77.38it/s]


In [32]:
save_path = r"E:\pUGTdb\20251128_collection\download_PF00201_taxonomy33090\step2_get_3D_Protein\smiles_molt5_embedding1.h5"
# 保存
with h5py.File(save_path, 'w') as f:
    for id_, embedding in molecule_embedding_molt5.items():
        f.create_dataset(id_, data=np.array(embedding))

# DeepChem embedding

In [33]:
def get_continuous_features(smiles_list):
    """
    提取高密度的浮点数特征 (Mordred via DeepChem)
    """
    print("--- 正在提取 Mordred 全浮点数特征 ---")
    
    # 1. 使用 Mordred Featurizer (ignore_3D=True 只计算 2D 拓扑特征，速度快)
    # 这会计算约 1613 个特征，包括 LogP, TPSA, 拓扑指数, 电子能量等
    featurizer = dc.feat.MordredDescriptors(ignore_3D=True)
    
    # 2. 提取特征
    try:
        features = featurizer.featurize(smiles_list)
    except ModuleNotFoundError:
        print("❌ 错误: 请先安装 mordred 库 (pip install mordred)")
        return None

    # 3. 数据清洗 (关键步骤！)
    valid_features = []
    feature_dim = 1613 # Mordred 默认维度
    
    for i, f in enumerate(features):
        # 处理提取失败的情况
        if f is None or f.size == 0:
            valid_features.append(np.zeros((feature_dim,), dtype=np.float32))
            continue
            
        # 替换 NaN (无效值) 和 Infinity (无穷大)
        # 浮点数描述符计算中经常会出现除以0的情况导致 NaN，必须填补
        clean_f = np.nan_to_num(f, nan=0.0, posinf=1000.0, neginf=-1000.0)
        clean_f = clean_f.reshape(1,-1)
        return clean_f.astype(np.float32)
    

In [34]:
# --- 准备测试数据 ---

molecule_embedding_deepchem = {}
for smiles, id_ in tqdm(smiles_to_id.items()):
    deepchem_feats = get_continuous_features(smiles)
    molecule_embedding_deepchem[id_] = deepchem_feats

  0%|          | 0/154 [00:00<?, ?it/s]

--- 正在提取 Mordred 全浮点数特征 ---


  1%|          | 1/154 [00:01<03:09,  1.24s/it]

--- 正在提取 Mordred 全浮点数特征 ---


  1%|▏         | 2/154 [00:02<03:08,  1.24s/it]

...

--- 正在提取 Mordred 全浮点数特征 ---
--- 正在提取 Mordred 全浮点数特征 ---


 99%|█████████▊| 152/154 [01:14<00:01,  1.45it/s]

--- 正在提取 Mordred 全浮点数特征 ---


100%|██████████| 154/154 [01:14<00:00,  2.06it/s]

--- 正在提取 Mordred 全浮点数特征 ---


In [35]:
save_path = r"E:\pUGTdb\20251128_collection\download_PF00201_taxonomy33090\step2_get_3D_Protein\smiles_deepchem_embedding1.h5"
# 保存
with h5py.File(save_path, 'w') as f:
    for id_, embedding in molecule_embedding_deepchem.items():
        f.create_dataset(id_, data=np.array(embedding))

# test load data

## deepchem

In [5]:
save_path = r"E:\pUGTdb\20251128_collection\download_PF00201_taxonomy33090\step2_get_3D_Protein\smiles_deepchem_embedding.h5"
h5_file = h5py.File(save_path, 'r')

In [6]:
h5_file.keys()

<KeysViewHDF5 ['SMILES_00001', 'SMILES_00002', 'SMILES_00003', 'SMILES_00004', 'SMILES_00005', 'SMILES_00006', 'SMILES_00007', 'SMILES_00008', 'SMILES_00009', 'SMILES_00010', 'SMILES_00011', 'SMILES_00012', 'SMILES_00013', 'SMILES_00014', 'SMILES_00015', 'SMILES_00016', 'SMILES_00017', 'SMILES_00018', 'SMILES_00019', 'SMILES_00020', 'SMILES_00021', 'SMILES_00022', 'SMILES_00023', 'SMILES_00024', 'SMILES_00025', 'SMILES_00026', 'SMILES_00027', 'SMILES_00028', 'SMILES_00029', 'SMILES_00030', 'SMILES_00031', 'SMILES_00032', 'SMILES_00033', 'SMILES_00034', 'SMILES_00035', 'SMILES_00036', 'SMILES_00037', 'SMILES_00038', 'SMILES_00039', 'SMILES_00040', 'SMILES_00041', 'SMILES_00042', 'SMILES_00043', 'SMILES_00044', 'SMILES_00045', 'SMILES_00046', 'SMILES_00047', 'SMILES_00048', 'SMILES_00049', 'SMILES_00050', 'SMILES_00051', 'SMILES_00052', 'SMILES_00053', 'SMILES_00054', 'SMILES_00055', 'SMILES_00056', 'SMILES_00057', 'SMILES_00058', 'SMILES_00059', 'SMILES_00060', 'SMILES_00061', 'SMILES_0

In [9]:
h5_file['SMILES_00001'][0][:20]

array([  0.        ,   0.        ,   0.        ,   0.        ,
        32.704163  ,   2.4888058 ,   4.9776115 ,  32.704163  ,
         1.2112653 ,   4.2079897 ,   4.655426  ,   0.17242318,
         2.5312853 , 158.76523   ,   5.880193  ,   6.0606785 ,
        12.        ,  12.        ,  30.        ,  27.        ],
      dtype=float32)

## molt5

In [34]:
save_path = r"E:\pUGTdb\20251128_collection\download_PF00201_taxonomy33090\step2_get_3D_Protein\smiles_molt5_embedding.h5"
h5_file = h5py.File(save_path, 'r')

In [35]:
h5_file.keys()

<KeysViewHDF5 ['SMILES_00001', 'SMILES_00002', 'SMILES_00003', 'SMILES_00004', 'SMILES_00005', 'SMILES_00006', 'SMILES_00007', 'SMILES_00008', 'SMILES_00009', 'SMILES_00010', 'SMILES_00011', 'SMILES_00012', 'SMILES_00013', 'SMILES_00014', 'SMILES_00015', 'SMILES_00016', 'SMILES_00017', 'SMILES_00018', 'SMILES_00019', 'SMILES_00020', 'SMILES_00021', 'SMILES_00022', 'SMILES_00023', 'SMILES_00024', 'SMILES_00025', 'SMILES_00026', 'SMILES_00027', 'SMILES_00028', 'SMILES_00029', 'SMILES_00030', 'SMILES_00031', 'SMILES_00032', 'SMILES_00033', 'SMILES_00034', 'SMILES_00035', 'SMILES_00036', 'SMILES_00037', 'SMILES_00038', 'SMILES_00039', 'SMILES_00040', 'SMILES_00041', 'SMILES_00042', 'SMILES_00043', 'SMILES_00044', 'SMILES_00045', 'SMILES_00046', 'SMILES_00047', 'SMILES_00048', 'SMILES_00049', 'SMILES_00050', 'SMILES_00051', 'SMILES_00052', 'SMILES_00053', 'SMILES_00054', 'SMILES_00055', 'SMILES_00056', 'SMILES_00057', 'SMILES_00058', 'SMILES_00059', 'SMILES_00060', 'SMILES_00061', 'SMILES_0

In [36]:
h5_file['SMILES_00001'].shape

(1, 512)

In [37]:
h5_file['SMILES_00001'][0][:20]

array([ 0.04094646, -0.05783999, -0.09128102, -0.02279633, -0.16071226,
       -0.09331299, -0.1171034 , -0.15611656, -0.01792923, -0.06701498,
       -0.1322251 , -0.16644423,  0.02098801, -0.19135895,  0.03224719,
       -0.2701135 , -0.02847478,  0.24505167,  0.46757153,  0.00451825],
      dtype=float32)

## UniMol

In [30]:
save_path = r"E:\pUGTdb\20251128_collection\download_PF00201_taxonomy33090\step2_get_3D_Protein\smiles_UniMol_embedding.h5"
h5_file = h5py.File(save_path, 'r')

In [31]:
h5_file.keys()

<KeysViewHDF5 ['SMILES_00001', 'SMILES_00002', 'SMILES_00003', 'SMILES_00004', 'SMILES_00005', 'SMILES_00006', 'SMILES_00007', 'SMILES_00008', 'SMILES_00009', 'SMILES_00010', 'SMILES_00011', 'SMILES_00012', 'SMILES_00013', 'SMILES_00014', 'SMILES_00015', 'SMILES_00016', 'SMILES_00017', 'SMILES_00018', 'SMILES_00019', 'SMILES_00020', 'SMILES_00021', 'SMILES_00022', 'SMILES_00023', 'SMILES_00024', 'SMILES_00025', 'SMILES_00026', 'SMILES_00027', 'SMILES_00028', 'SMILES_00029', 'SMILES_00030', 'SMILES_00031', 'SMILES_00032', 'SMILES_00033', 'SMILES_00034', 'SMILES_00035', 'SMILES_00036', 'SMILES_00037', 'SMILES_00038', 'SMILES_00039', 'SMILES_00040', 'SMILES_00041', 'SMILES_00042', 'SMILES_00043', 'SMILES_00044', 'SMILES_00045', 'SMILES_00046', 'SMILES_00047', 'SMILES_00048', 'SMILES_00049', 'SMILES_00050', 'SMILES_00051', 'SMILES_00052', 'SMILES_00053', 'SMILES_00054', 'SMILES_00055', 'SMILES_00056', 'SMILES_00057', 'SMILES_00058', 'SMILES_00059', 'SMILES_00060', 'SMILES_00061', 'SMILES_0

In [32]:
h5_file['SMILES_00001'].shape

(1, 512)

In [33]:
h5_file['SMILES_00001'][0][:20]

array([ 0.5376201 ,  0.30387986, -0.14941293, -0.46971023,  0.9416121 ,
       -2.304003  ,  0.4969352 ,  0.16281036,  0.59448177,  1.6861506 ,
       -0.05139968,  1.3682148 ,  0.16981252, -0.51476836, -2.5095067 ,
        0.3131239 , -0.19105874, -0.36948815,  0.14025418, -1.7760587 ],
      dtype=float32)

## RDKit

In [207]:
save_path = r"E:\pUGTdb\20251128_collection\download_PF00201_taxonomy33090\step2_get_3D_Protein\smiles_RDKit_embedding.h5"
h5_file = h5py.File(save_path, 'r')

In [208]:
h5_file.keys()

<KeysViewHDF5 ['SMILES_00001', 'SMILES_00002', 'SMILES_00003', 'SMILES_00004', 'SMILES_00005', 'SMILES_00006', 'SMILES_00007', 'SMILES_00008', 'SMILES_00009', 'SMILES_00010', 'SMILES_00011', 'SMILES_00012', 'SMILES_00013', 'SMILES_00014', 'SMILES_00015', 'SMILES_00016', 'SMILES_00017', 'SMILES_00018', 'SMILES_00019', 'SMILES_00020', 'SMILES_00021', 'SMILES_00022', 'SMILES_00023', 'SMILES_00024', 'SMILES_00025', 'SMILES_00026', 'SMILES_00027', 'SMILES_00028', 'SMILES_00029', 'SMILES_00030', 'SMILES_00031', 'SMILES_00032', 'SMILES_00033', 'SMILES_00034', 'SMILES_00035', 'SMILES_00036', 'SMILES_00037', 'SMILES_00038', 'SMILES_00039', 'SMILES_00040', 'SMILES_00041', 'SMILES_00042', 'SMILES_00043', 'SMILES_00044', 'SMILES_00045', 'SMILES_00046', 'SMILES_00047', 'SMILES_00048', 'SMILES_00049', 'SMILES_00050', 'SMILES_00051', 'SMILES_00052', 'SMILES_00053', 'SMILES_00054', 'SMILES_00055', 'SMILES_00056', 'SMILES_00057', 'SMILES_00058', 'SMILES_00059', 'SMILES_00060', 'SMILES_00061', 'SMILES_0

In [209]:
h5_file['SMILES_00001'].shape

(1, 2048)

In [31]:



import os
import numpy as np
from biotite.structure.io import pdb
import biotite.structure as struc

def calculate_rg(pdb_path):
    """计算单个 PDB 的回转半径 (Rg)"""
    try:
        # 1. 读取 PDB 文件
        file = pdb.PDBFile.read(pdb_path)

        # 选择第一个模型（如果有多个模型）
        if file.get_model_count() > 0:
            array = file.get_structure(model=1)
        else:
            array = file.get_structure()

        # 2. 筛选 C-alpha 原子 (骨架)
        ca_mask = array.atom_name == "CA"
        target_atoms = array[ca_mask]

        if target_atoms.array_length() == 0:
            print(f"警告: {os.path.basename(pdb_path)} 中没有 C-alpha 原子")
            return None
        
        # 3. 计算回转半径 (Rg)
        rg = struc.gyration_radius(target_atoms)
        return rg

    except Exception as e:
        print(f"计算出错 {os.path.basename(pdb_path)}: {e}")
        return None

def calculate_plddt(pdb_path):
    """计算单个 PDB 的 pLDDT"""
    try:
        # 1. 读取 PDB 文件
        file = pdb.PDBFile.read(pdb_path)

        # 显式加载 b_factor 列
        if file.get_model_count() > 0:
            array = file.get_structure(model=1, extra_fields=['b_factor'])
        else:
            array = file.get_structure(extra_fields=['b_factor'])

        # 检查是否加载了 b_factor
        if not hasattr(array, 'b_factor'):
            print(f"警告: {os.path.basename(pdb_path)} 未发现 b_factor 属性")
            return None

        # 2. 筛选 C-alpha 原子
        ca_mask = array.atom_name == "CA"
        target_atoms = array[ca_mask]

        if target_atoms.array_length() == 0:
            target_atoms = array  # 如果没有 C-alpha，使用所有原子
        
        # 3. 计算 pLDDT（平均 B-factor）
        mean_plddt = np.mean(target_atoms.b_factor)
        return mean_plddt

    except Exception as e:
        print(f"解析失败 {os.path.basename(pdb_path)}: {str(e)}")
        return None

# 示例：计算某个 PDB 文件的 Rg 和 pLDDT
pdb_file_path = r"F:\反应条件生成对比实验\REXzyme\generated\OmegaFold\RHEA_23164 _ 5 _ EC 2.4.1.237 _ score=13.3705 _ len=497.pdb"

rg = calculate_rg(pdb_file_path)
plddt = calculate_plddt(pdb_file_path)

if rg is not None:
    print(f"回转半径 (Rg): {rg} Å")
if plddt is not None:
    print(f"pLDDT: {plddt}")


回转半径 (Rg): 33.09611361693622 Å
pLDDT: 37.445682281059064


In [ ]:
for t in 

In [41]:
import os

pdb_file_path = r"C:\Users\admin\Desktop\tc"

# 遍历文件夹中的所有文件
for k in os.listdir(pdb_file_path):
    k_path = os.path.join(pdb_file_path, k)
    
    # 检查是否是文件，并且文件扩展名为 .pdb
    if os.path.isfile(k_path) and k_path.lower().endswith(".pdb"):
        print(f"正在处理文件: {k_path}")

        # 计算回转半径和 pLDDT
        rg = calculate_rg(k_path)
        plddt = calculate_plddt(k_path)
        
        if rg is not None:
            print(f"回转半径 (Rg): {rg} Å")
        if plddt is not None:
            print(f"pLDDT: {plddt}")
    else:
        print(f"跳过非 PDB 文件或文件夹: {k_path}")


正在处理文件: C:\Users\admin\Desktop\tc\AF-Q94C57-F1-model_v6.pdb
回转半径 (Rg): 22.109384315945075 Å
pLDDT: 92.50093167701863
